# 02 - Data Cleaning

Strict cleaning notebook for the combined CPCB/CAAQMS Indian air-quality time-series dataset.

This notebook prepares one validated interim Parquet file. It does **not** calculate AQI, encode geography, create temporal features, interpolate pollutant values, or build forecasting features; those steps belong in later notebooks.


## 1. Imports and Display Settings


In [1]:
from pathlib import Path
import re
import sys

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)

print(f"Python version: {sys.version.split()[0]}")
print(f"pandas version: {pd.__version__}")


Python version: 3.14.4
pandas version: 3.0.3


## 2. Project Paths


In [2]:
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "Data"
RAW_DATA_DIR = DATA_DIR / "Raw"
INTERIM_DATA_DIR = DATA_DIR / "Interim"

DATA_FILE = RAW_DATA_DIR / "AQI_Dataset.csv"

OUTPUT_FILE = (
    INTERIM_DATA_DIR /
    "cleaned_air_quality.parquet"
)

print(f"Raw input: {DATA_FILE}")
print(f"Cleaned output: {OUTPUT_FILE}")


Raw input: ../Data/Raw/AQI_Dataset.csv
Cleaned output: ../Data/Interim/cleaned_air_quality.parquet


## 3. Load Raw Combined Dataset


In [3]:
if not DATA_FILE.exists():
    raise FileNotFoundError(f"Raw data file not found: {DATA_FILE}")

df = pd.read_csv(DATA_FILE, low_memory=False)

print("Dataset loaded successfully")
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]:,}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:,.2f} MB")


Dataset loaded successfully
Rows: 3,431,900
Columns: 25
Memory usage: 832.16 MB


## 4. Initial Inspection


In [4]:
print("Raw columns:")
for col in df.columns:
    print(f"- {col}")

print("\nRaw dtypes:")
print(df.dtypes)

print("\nFirst five rows:")
print(df.head().to_string())


Raw columns:
- Timestamp
- PM2.5
- PM10
- Nitric Oxide
- Nitrogen Dioxide
- Nitrogen Oxides
- Ammonia
- Sulfur Dioxide
- Carbon Monoxide
- Ozone
- Ambient Temperature
- Relative Humidity
- Solar Radiation
- Rainfall
- State
- City
- Latitude
- Longitude
- Calculated_AQI
- Month
- Hour
- DayOfWeek
- Is_Weekend
- State_Encoded
- City_Encoded

Raw dtypes:
Timestamp                  str
PM2.5                  float64
PM10                   float64
Nitric Oxide           float64
Nitrogen Dioxide       float64
Nitrogen Oxides        float64
Ammonia                float64
Sulfur Dioxide         float64
Carbon Monoxide        float64
Ozone                  float64
Ambient Temperature    float64
Relative Humidity      float64
Solar Radiation        float64
Rainfall               float64
State                      str
City                       str
Latitude               float64
Longitude              float64
Calculated_AQI         float64
Month                    int64
Hour                     

## 5. Standardize Column Names


In [5]:
def standardize_column_name(column: str) -> str:
    """Convert raw column labels to stable snake_case names."""
    name = str(column).strip().lower()
    name = name.replace("µ", "u")
    name = re.sub(r"[^0-9a-zA-Z]+", "_", name)
    name = re.sub(r"_+", "_", name)
    return name.strip("_")

original_columns = list(df.columns)
df.columns = [standardize_column_name(col) for col in df.columns]

column_mapping_preview = pd.DataFrame({
    "raw_column": original_columns,
    "standardized_column": df.columns,
})
print(column_mapping_preview.to_string(index=False))

duplicate_columns = pd.Index(df.columns)[pd.Index(df.columns).duplicated()].unique().tolist()
if duplicate_columns:
    raise ValueError(f"Column standardization created duplicate names: {duplicate_columns}")


         raw_column standardized_column
          Timestamp           timestamp
              PM2.5               pm2_5
               PM10                pm10
       Nitric Oxide        nitric_oxide
   Nitrogen Dioxide    nitrogen_dioxide
    Nitrogen Oxides     nitrogen_oxides
            Ammonia             ammonia
     Sulfur Dioxide      sulfur_dioxide
    Carbon Monoxide     carbon_monoxide
              Ozone               ozone
Ambient Temperature ambient_temperature
  Relative Humidity   relative_humidity
    Solar Radiation     solar_radiation
           Rainfall            rainfall
              State               state
               City                city
           Latitude            latitude
          Longitude           longitude
     Calculated_AQI      calculated_aqi
              Month               month
               Hour                hour
          DayOfWeek           dayofweek
         Is_Weekend          is_weekend
      State_Encoded       state_encoded


## 6. Standardize Pollutant Columns and Remove Pre-Engineered Features


In [6]:
POLLUTANT_RENAME_MAP = {
    "pm25": "pm2_5",
    "pm2_5": "pm2_5",
    "particulate_matter_2_5": "pm2_5",
    "particulate_matter_25": "pm2_5",
    "pm10": "pm10",
    "nitric_oxide": "no",
    "nitrogen_dioxide": "no2",
    "nitrogen_oxides": "nox",
    "ammonia": "nh3",
    "sulfur_dioxide": "so2",
    "sulphur_dioxide": "so2",
    "carbon_monoxide": "co",
    "ozone": "o3",
}

renamed_pollutants = {}
for source, target in POLLUTANT_RENAME_MAP.items():
    if source not in df.columns or source == target:
        continue

    if target in df.columns:
        # If both a descriptive and canonical pollutant column exist, preserve non-null canonical values first.
        df[target] = df[target].combine_first(df[source])
        df = df.drop(columns=[source])
        renamed_pollutants[source] = f"combined into existing {target}"
    else:
        df = df.rename(columns={source: target})
        renamed_pollutants[source] = target

POLLUTANT_COLUMNS = [
    "pm2_5",
    "pm10",
    "no",
    "no2",
    "nox",
    "nh3",
    "co",
    "so2",
    "o3",
]

available_pollutants = [col for col in POLLUTANT_COLUMNS if col in df.columns]
missing_pollutants = [col for col in POLLUTANT_COLUMNS if col not in df.columns]

print("Pollutant rename actions:")
print(renamed_pollutants if renamed_pollutants else "No descriptive pollutant columns required renaming")
print(f"\nAvailable pollutants ({len(available_pollutants)}): {available_pollutants}")
print(f"Missing pollutants ({len(missing_pollutants)}): {missing_pollutants}")

if not available_pollutants:
    raise ValueError("No pollutant columns were found after column standardization.")

ENGINEERED_COLUMNS = [
    "calculated_aqi",
    "month",
    "hour",
    "dayofweek",
    "is_weekend",
    "state_encoded",
    "city_encoded",
]

existing_engineered_columns = [col for col in ENGINEERED_COLUMNS if col in df.columns]
if existing_engineered_columns:
    df = df.drop(columns=existing_engineered_columns)

print(f"\nRemoved engineered columns: {existing_engineered_columns}")
print(f"Remaining columns: {df.shape[1]:,}")


Pollutant rename actions:
{'nitric_oxide': 'no', 'nitrogen_dioxide': 'no2', 'nitrogen_oxides': 'nox', 'ammonia': 'nh3', 'sulfur_dioxide': 'so2', 'carbon_monoxide': 'co', 'ozone': 'o3'}

Available pollutants (9): ['pm2_5', 'pm10', 'no', 'no2', 'nox', 'nh3', 'co', 'so2', 'o3']
Missing pollutants (0): []

Removed engineered columns: ['calculated_aqi', 'month', 'hour', 'dayofweek', 'is_weekend', 'state_encoded', 'city_encoded']
Remaining columns: 18


## 7. Timestamp Column Detection and Raw Timestamp Audit


In [7]:
timestamp_candidates = [
    "timestamp",
    "datetime",
    "date_time",
    "date",
    "from_date",
    "sampling_date",
]

available_timestamp_candidates = [col for col in timestamp_candidates if col in df.columns]
if not available_timestamp_candidates:
    raise ValueError("No timestamp-like column found in the dataset.")

if "timestamp" not in df.columns:
    timestamp_col = available_timestamp_candidates[0]
    df = df.rename(columns={timestamp_col: "timestamp"})
    print(f"Renamed timestamp column '{timestamp_col}' to 'timestamp'")
else:
    print("Using existing 'timestamp' column")

timestamp_raw = df["timestamp"].astype("string").str.strip()

print("Raw timestamp samples from the start:")
print(timestamp_raw.head(10).to_string(index=False))

print("\nRaw timestamp samples from the end:")
print(timestamp_raw.tail(10).to_string(index=False))

print("\nRepresentative random raw timestamp samples:")
print(timestamp_raw.dropna().sample(min(20, timestamp_raw.notna().sum()), random_state=42).to_string(index=False))

raw_2026_mask = timestamp_raw.str.contains("2026", na=False, regex=False)
print(f"\nRaw timestamp values containing '2026': {raw_2026_mask.sum():,}")
if raw_2026_mask.any():
    print("First raw 2026 timestamp values:")
    print(timestamp_raw[raw_2026_mask].head(20).to_string(index=False))
    print("Last raw 2026 timestamp values:")
    print(timestamp_raw[raw_2026_mask].tail(20).to_string(index=False))


Using existing 'timestamp' column
Raw timestamp samples from the start:
2017-09-05 11:00:00
2017-09-05 12:00:00
2017-09-05 13:00:00
2017-09-05 14:00:00
2017-09-05 15:00:00
2017-09-05 16:00:00
2017-09-05 17:00:00
2017-09-05 18:00:00
2017-09-05 19:00:00
2017-09-05 20:00:00

Raw timestamp samples from the end:
2026-06-30 14:00:00
2026-06-30 15:00:00
2026-06-30 16:00:00
2026-06-30 17:00:00
2026-06-30 18:00:00
2026-06-30 19:00:00
2026-06-30 20:00:00
2026-06-30 21:00:00
2026-06-30 22:00:00
2026-06-30 23:00:00

Representative random raw timestamp samples:
2023-11-28 21:00:00
2019-07-20 16:00:00
2023-03-12 05:00:00
2019-06-07 06:00:00
2024-06-17 04:00:00
2022-04-20 16:00:00
2023-07-20 11:00:00
2025-04-10 15:00:00
2024-06-05 16:00:00
2022-11-04 09:00:00
2024-09-02 03:00:00
2019-03-13 21:00:00
2021-03-03 00:00:00
2025-05-16 13:00:00
2019-08-20 02:00:00
2025-05-04 17:00:00
2018-05-12 16:00:00
2023-02-09 08:00:00
2020-02-26 06:00:00
2021-09-23 11:00:00

Raw timestamp values containing '2026': 219,

## 8. Deterministic Timestamp Parsing

The raw dataset stores dates as `YYYY-MM-DD HH:MM:SS`. The parser below first uses exact year-first formats, then only parses day/month-first numeric formats when the order is inferable from values where one date component exceeds 12. Ambiguous `DD-MM-YYYY` versus `MM-DD-YYYY` values are not silently guessed.


In [8]:
def _assign_exact_datetime_format(raw: pd.Series, parsed: pd.Series, label: str, pattern: str, fmt: str, parse_report: list) -> pd.Series:
    mask = raw.str.fullmatch(pattern, na=False) & parsed.isna()
    if not mask.any():
        parse_report.append({"format": label, "matched_rows": 0, "parse_failures": 0})
        return parsed

    parsed_values = pd.to_datetime(raw.loc[mask], format=fmt, errors="coerce")
    parsed.loc[mask] = parsed_values
    parse_report.append({
        "format": label,
        "matched_rows": int(mask.sum()),
        "parse_failures": int(parsed_values.isna().sum()),
    })
    return parsed


def _parse_day_month_year_formats(raw: pd.Series, parsed: pd.Series, parse_report: list) -> pd.Series:
    time_variants = [
        (r"(?:[ T])\d{2}:\d{2}:\d{2}", " %H:%M:%S", "with seconds"),
        (r"(?:[ T])\d{2}:\d{2}", " %H:%M", "with minutes"),
        (r"", "", "date only"),
    ]
    separators = [("-", r"-", "dash"), ("/", r"/", "slash")]

    for sep_char, sep_regex, sep_label in separators:
        for time_regex, time_fmt, time_label in time_variants:
            pattern = rf"\d{{2}}{sep_regex}\d{{2}}{sep_regex}\d{{4}}{time_regex}"
            mask = raw.str.fullmatch(pattern, na=False) & parsed.isna()
            if not mask.any():
                continue

            extract_pattern = rf"^(?P<first>\d{{2}}){sep_regex}(?P<second>\d{{2}}){sep_regex}(?P<year>\d{{4}})"
            parts = raw.loc[mask].str.extract(extract_pattern).astype("Int64")
            first_gt_12 = int((parts["first"] > 12).sum())
            second_gt_12 = int((parts["second"] > 12).sum())

            if first_gt_12 and not second_gt_12:
                fmt = f"%d{sep_char}%m{sep_char}%Y{time_fmt}"
                label = f"DD{sep_char}MM{sep_char}YYYY {time_label}"
            elif second_gt_12 and not first_gt_12:
                fmt = f"%m{sep_char}%d{sep_char}%Y{time_fmt}"
                label = f"MM{sep_char}DD{sep_char}YYYY {time_label}"
            elif not first_gt_12 and not second_gt_12:
                examples = raw.loc[mask].drop_duplicates().head(10).tolist()
                raise ValueError(
                    "Ambiguous day/month timestamp format detected. "
                    f"Cannot safely infer DD{sep_char}MM{sep_char}YYYY versus MM{sep_char}DD{sep_char}YYYY. "
                    f"Examples: {examples}"
                )
            else:
                examples = raw.loc[mask].drop_duplicates().head(10).tolist()
                raise ValueError(
                    "Mixed incompatible day/month timestamp patterns detected. "
                    f"Examples: {examples}"
                )

            parsed_values = pd.to_datetime(raw.loc[mask], format=fmt, errors="coerce")
            parsed.loc[mask] = parsed_values
            parse_report.append({
                "format": label,
                "matched_rows": int(mask.sum()),
                "parse_failures": int(parsed_values.isna().sum()),
            })

    return parsed


def parse_timestamp_series(raw: pd.Series) -> tuple[pd.Series, pd.DataFrame]:
    raw = raw.astype("string").str.strip()
    parsed = pd.Series(pd.NaT, index=raw.index, dtype="datetime64[ns]")
    parse_report = []

    exact_year_first_formats = [
        ("YYYY-MM-DD HH:MM:SS", r"\d{4}-\d{2}-\d{2}(?:[ T])\d{2}:\d{2}:\d{2}", "%Y-%m-%d %H:%M:%S"),
        ("YYYY-MM-DD HH:MM", r"\d{4}-\d{2}-\d{2}(?:[ T])\d{2}:\d{2}", "%Y-%m-%d %H:%M"),
        ("YYYY-MM-DD", r"\d{4}-\d{2}-\d{2}", "%Y-%m-%d"),
        ("YYYY/MM/DD HH:MM:SS", r"\d{4}/\d{2}/\d{2}(?:[ T])\d{2}:\d{2}:\d{2}", "%Y/%m/%d %H:%M:%S"),
        ("YYYY/MM/DD HH:MM", r"\d{4}/\d{2}/\d{2}(?:[ T])\d{2}:\d{2}", "%Y/%m/%d %H:%M"),
        ("YYYY/MM/DD", r"\d{4}/\d{2}/\d{2}", "%Y/%m/%d"),
    ]

    for label, pattern, fmt in exact_year_first_formats:
        parsed = _assign_exact_datetime_format(raw, parsed, label, pattern, fmt, parse_report)

    parsed = _parse_day_month_year_formats(raw, parsed, parse_report)

    unmatched_mask = raw.notna() & parsed.isna()
    if unmatched_mask.any():
        parse_report.append({
            "format": "unmatched_or_invalid",
            "matched_rows": int(unmatched_mask.sum()),
            "parse_failures": int(unmatched_mask.sum()),
        })

    return parsed, pd.DataFrame(parse_report)

parsed_timestamp, timestamp_parse_report = parse_timestamp_series(timestamp_raw)
print("Timestamp format detection and parsing report:")
print(timestamp_parse_report.to_string(index=False))

df["timestamp"] = parsed_timestamp

invalid_timestamp_count = int(df["timestamp"].isna().sum())
invalid_timestamp_pct = invalid_timestamp_count / len(df) * 100
print(f"\nInvalid timestamps after deterministic parsing: {invalid_timestamp_count:,} ({invalid_timestamp_pct:.4f}%)")

if invalid_timestamp_count:
    print("Invalid raw timestamp examples:")
    print(timestamp_raw[df["timestamp"].isna()].drop_duplicates().head(20).to_string(index=False))

print(f"Parsed timestamp minimum: {df['timestamp'].min()}")
print(f"Parsed timestamp maximum: {df['timestamp'].max()}")

print("\nRows by year-month after parsing:")
year_month_counts = df.loc[df["timestamp"].notna(), "timestamp"].dt.to_period("M").value_counts().sort_index()
print(year_month_counts.to_string())

NOTEBOOK_EXECUTION_DATE = pd.Timestamp.now().normalize()
DATA_COLLECTION_CUTOFF = NOTEBOOK_EXECUTION_DATE
future_timestamp_mask = df["timestamp"] > DATA_COLLECTION_CUTOFF
print(f"\nNotebook execution date cutoff: {DATA_COLLECTION_CUTOFF}")
print(f"Timestamps later than cutoff: {future_timestamp_mask.sum():,}")
if future_timestamp_mask.any():
    print(df.loc[future_timestamp_mask, "timestamp"].sort_values().drop_duplicates().head(20).to_string(index=False))


Timestamp format detection and parsing report:
             format  matched_rows  parse_failures
YYYY-MM-DD HH:MM:SS       3431900               0
   YYYY-MM-DD HH:MM             0               0
         YYYY-MM-DD             0               0
YYYY/MM/DD HH:MM:SS             0               0
   YYYY/MM/DD HH:MM             0               0
         YYYY/MM/DD             0               0

Invalid timestamps after deterministic parsing: 0 (0.0000%)
Parsed timestamp minimum: 2017-01-01 00:00:00
Parsed timestamp maximum: 2026-06-30 23:00:00

Rows by year-month after parsing:
timestamp
2017-01     1584
2017-02     1949
2017-03     3256
2017-04     2445
2017-05     2964
2017-06     2363
2017-07     2934
2017-08     2936
2017-09     3703
2017-10     6407
2017-11     6991
2017-12     6631
2018-01     8081
2018-02    10364
2018-03    11277
2018-04     9858
2018-05    10644
2018-06    11059
2018-07    12609
2018-08    12724
2018-09    12411
2018-10    13524
2018-11    12726
2018-12    143

## 9. Remove Invalid Timestamps


In [9]:
rows_before = len(df)
df = df.dropna(subset=["timestamp"]).copy()
rows_removed = rows_before - len(df)

print(f"Rows removed because timestamp could not be parsed: {rows_removed:,}")
print(f"Rows remaining: {len(df):,}")


Rows removed because timestamp could not be parsed: 0
Rows remaining: 3,431,900


## 10. Convert Pollutants and Weather Columns to Numeric


In [10]:
for col in available_pollutants:
    df[col] = pd.to_numeric(df[col], errors="coerce")

WEATHER_COLUMNS = [
    "ambient_temperature",
    "relative_humidity",
    "solar_radiation",
    "rainfall",
]
available_weather_columns = [col for col in WEATHER_COLUMNS if col in df.columns]

for col in available_weather_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(f"Numeric pollutant columns: {available_pollutants}")
print(f"Numeric weather columns preserved: {available_weather_columns}")


Numeric pollutant columns: ['pm2_5', 'pm10', 'no', 'no2', 'nox', 'nh3', 'co', 'so2', 'o3']
Numeric weather columns preserved: ['ambient_temperature', 'relative_humidity', 'solar_radiation', 'rainfall']


## 11. Replace Infinite Values


In [11]:
pollutant_infinite_before = np.isinf(df[available_pollutants]).sum()
print("Infinite pollutant values before replacement:")
print(pollutant_infinite_before.to_string())

df[available_pollutants] = df[available_pollutants].replace([np.inf, -np.inf], np.nan)

if available_weather_columns:
    weather_infinite_before = np.isinf(df[available_weather_columns]).sum()
    print("\nInfinite weather values before replacement:")
    print(weather_infinite_before.to_string())
    df[available_weather_columns] = df[available_weather_columns].replace([np.inf, -np.inf], np.nan)


Infinite pollutant values before replacement:
pm2_5    0
pm10     0
no       0
no2      0
nox      0
nh3      0
co       0
so2      0
o3       0

Infinite weather values before replacement:
ambient_temperature    0
relative_humidity      0
solar_radiation        0
rainfall               0


## 12. Remove Physically Invalid Pollutant Values

Negative concentrations and values above broad data-quality sanity limits are treated as invalid measurements and set to missing. These limits are not AQI breakpoints, and no IQR outlier deletion is performed so genuine severe pollution episodes remain available for modeling.


In [12]:
POLLUTANT_MAX_LIMITS = {
    "pm2_5": 2000,
    "pm10": 3000,
    "no": 1000,
    "no2": 1000,
    "nox": 2000,
    "nh3": 2000,
    "co": 100,
    "so2": 2000,
    "o3": 1000,
}

negative_counts = (df[available_pollutants] < 0).sum()
print("Negative pollutant values before cleaning:")
print(negative_counts.to_string())

for col in available_pollutants:
    df.loc[df[col] < 0, col] = np.nan

above_limit_counts = {}
for col in available_pollutants:
    max_limit = POLLUTANT_MAX_LIMITS[col]
    above_limit_mask = df[col] > max_limit
    above_limit_counts[col] = int(above_limit_mask.sum())
    df.loc[above_limit_mask, col] = np.nan

print("\nPollutant values above sanity limits before cleaning:")
print(pd.Series(above_limit_counts).to_string())

print("\nPollutant maxima after sanity cleaning:")
print(df[available_pollutants].max().to_string())


Negative pollutant values before cleaning:
pm2_5    0
pm10     0
no       0
no2      0
nox      0
nh3      0
co       0
so2      0
o3       0

Pollutant values above sanity limits before cleaning:
pm2_5    0
pm10     0
no       0
no2      0
nox      0
nh3      0
co       0
so2      0
o3       0

Pollutant maxima after sanity cleaning:
pm2_5     999.99
pm10     1000.00
no        500.00
no2       499.99
nox       500.00
nh3       499.99
co         41.67
so2       200.00
o3        932.00


## 13. Clean State and City Names


In [13]:
STATE_NAME_CORRECTIONS = {
    "Harayana": "Haryana",
}

for col in ["state", "city"]:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype("string")
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .str.title()
        )
        blank_mask = df[col].eq("")
        df.loc[blank_mask, col] = pd.NA

        if col == "state":
            correction_counts = {
                f"{source} -> {target}": int(df[col].eq(source).sum())
                for source, target in STATE_NAME_CORRECTIONS.items()
                if int(df[col].eq(source).sum()) > 0
            }
            df[col] = df[col].replace(STATE_NAME_CORRECTIONS)
            print("State name corrections applied:")
            print(correction_counts if correction_counts else "No known state spelling corrections were needed")

        print(f"{col}: {df[col].nunique(dropna=True):,} unique non-null values")
        print(df[col].dropna().sort_values().drop_duplicates().head(25).to_string(index=False))
    else:
        print(f"Column '{col}' not present")


State name corrections applied:
{'Harayana -> Haryana': 267875}
state: 19 unique non-null values
   Andhra Pradesh
Arunachal Pradesh
            Assam
            Bihar
       Chandigarh
     Chhattisgarh
            Delhi
          Gujarat
          Haryana
 Himachal Pradesh
        Jharkhand
        Karnataka
           Kerala
   Madhya Pradesh
      Maharashtra
           Punjab
        Telangana
    Uttar Pradesh
      West Bengal


city: 66 unique non-null values
                   32Bungalows, Bhilai (, )
                         Aiims, Raipur (, )
                                Alipur (, )
Anand Kala Kshetram, Rajamahendravaram (, )
                          Anand Vihar ( , )
               Arya Nagar, Bahadurgarh (, )
           Asansol Court Area, Asansol (, )
                           Ashok Vihar (, )
                             Aya Nagar (, )
                   Ballygunge, Kolkata (, )
                  Bandra Kurla Complex (, )
                                Bawana (, )
                    Belur Math, Howrah (, )
   Bollaram Industrial Area, Hyderabad (, )
                Btm Layout, Bengaluru ( , )
              Chhoti Gwaltoli, Indore ( , )
                  City Center, Gwalior (, )
                 Civil Line, Jalandhar (, )
                 Civil Lines, Bareilly (, )
         Dm Office_Kasipur, Samastipur (, )
                       Gidc, Nandesari (, )
              Girls College, Sivasagar (, )


## 14. Validate Latitude and Longitude


In [14]:
coordinate_columns = [col for col in ["latitude", "longitude"] if col in df.columns]
for col in coordinate_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

coordinate_invalid_counts = {}
if "latitude" in df.columns:
    invalid_latitude_mask = df["latitude"].notna() & ~df["latitude"].between(5, 40)
    coordinate_invalid_counts["latitude_outside_5_40"] = int(invalid_latitude_mask.sum())
    df.loc[invalid_latitude_mask, "latitude"] = np.nan

if "longitude" in df.columns:
    invalid_longitude_mask = df["longitude"].notna() & ~df["longitude"].between(65, 100)
    coordinate_invalid_counts["longitude_outside_65_100"] = int(invalid_longitude_mask.sum())
    df.loc[invalid_longitude_mask, "longitude"] = np.nan

print("Coordinate columns converted:", coordinate_columns)
print("Invalid coordinate counts set to NaN:")
print(pd.Series(coordinate_invalid_counts, dtype="int64").to_string() if coordinate_invalid_counts else "No coordinate columns found")

if coordinate_columns:
    print("\nCoordinate summary after validation:")
    print(df[coordinate_columns].describe().to_string())


Coordinate columns converted: ['latitude', 'longitude']
Invalid coordinate counts set to NaN:
latitude_outside_5_40       0
longitude_outside_65_100    0

Coordinate summary after validation:
           latitude     longitude
count  3.431900e+06  3.431900e+06
mean   2.309603e+01  7.895783e+01
std    5.934268e+00  4.836318e+00
min    8.878700e+00  7.259830e+01
25%    1.925250e+01  7.586960e+01
50%    2.368530e+01  7.731797e+01
75%    2.842800e+01  8.051817e+01
max    3.162013e+01  9.464040e+01


## 15. Weather Diagnostics and Basic Physical Checks


In [15]:
weather_cleaning_counts = {}

if "relative_humidity" in df.columns:
    mask = df["relative_humidity"].notna() & ~df["relative_humidity"].between(0, 100)
    weather_cleaning_counts["relative_humidity_outside_0_100"] = int(mask.sum())
    df.loc[mask, "relative_humidity"] = np.nan

if "solar_radiation" in df.columns:
    mask = df["solar_radiation"].notna() & (df["solar_radiation"] < 0)
    weather_cleaning_counts["negative_solar_radiation"] = int(mask.sum())
    df.loc[mask, "solar_radiation"] = np.nan

if "rainfall" in df.columns:
    mask = df["rainfall"].notna() & (df["rainfall"] < 0)
    weather_cleaning_counts["negative_rainfall"] = int(mask.sum())
    df.loc[mask, "rainfall"] = np.nan

if "ambient_temperature" in df.columns:
    mask = df["ambient_temperature"].notna() & ~df["ambient_temperature"].between(-50, 60)
    weather_cleaning_counts["ambient_temperature_outside_minus50_60"] = int(mask.sum())
    df.loc[mask, "ambient_temperature"] = np.nan

print("Weather values set to NaN by basic physical checks:")
print(pd.Series(weather_cleaning_counts, dtype="int64").to_string() if weather_cleaning_counts else "No weather columns found")

if available_weather_columns:
    weather_diagnostics = pd.DataFrame({
        "missing_count": df[available_weather_columns].isna().sum(),
        "missing_pct": df[available_weather_columns].isna().mean() * 100,
        "min": df[available_weather_columns].min(),
        "max": df[available_weather_columns].max(),
    })
    print("\nWeather diagnostics after basic checks:")
    print(weather_diagnostics.to_string())


Weather values set to NaN by basic physical checks:
relative_humidity_outside_0_100            0
negative_solar_radiation                   0
negative_rainfall                          0
ambient_temperature_outside_minus50_60    10



Weather diagnostics after basic checks:
                     missing_count  missing_pct   min     max
ambient_temperature             10     0.000291 -45.0    60.0
relative_humidity                0     0.000000   0.0   100.0
solar_radiation                  0     0.000000   0.0  1995.0
rainfall                         0     0.000000  -0.0    49.5


## 16. Station Identity Strategy


In [16]:
station_name_candidates = [
    col for col in df.columns
    if any(token in col for token in ["station", "location", "site", "monitor"])
]
print(f"Station/location-like columns available for inspection: {station_name_candidates}")

preferred_station_keys = ["state", "city", "latitude", "longitude"]
if all(col in df.columns for col in preferred_station_keys):
    STATION_KEYS = preferred_station_keys
elif all(col in df.columns for col in ["latitude", "longitude"]):
    STATION_KEYS = ["latitude", "longitude"]
else:
    raise ValueError("Cannot define station identity because latitude/longitude are unavailable.")

print(f"Final station identifier columns used: {STATION_KEYS}")
print(f"Estimated station count before duplicate aggregation: {df[STATION_KEYS].drop_duplicates().shape[0]:,}")


Station/location-like columns available for inspection: []
Final station identifier columns used: ['state', 'city', 'latitude', 'longitude']
Estimated station count before duplicate aggregation: 66


## 17. Remove Exact Duplicate Rows


In [17]:
exact_duplicate_rows = int(df.duplicated().sum())
print(f"Exact duplicate rows before removal: {exact_duplicate_rows:,}")

rows_before = len(df)
df = df.drop_duplicates().copy()
print(f"Exact duplicate rows removed: {rows_before - len(df):,}")
print(f"Rows remaining: {len(df):,}")


Exact duplicate rows before removal: 2,351


Exact duplicate rows removed: 2,351
Rows remaining: 3,429,549


## 18. Aggregate Duplicate Station-Timestamp Observations

Duplicate observations for the same station and timestamp are consolidated rather than blindly keeping the first row. Pollutants and numeric weather/sensor fields are averaged; stable categorical/location metadata uses the first non-null value within the duplicate group.


In [18]:
duplicate_key_columns = STATION_KEYS + ["timestamp"]
station_timestamp_duplicate_mask = df.duplicated(subset=duplicate_key_columns, keep=False)
duplicate_observation_rows = int(station_timestamp_duplicate_mask.sum())
duplicate_observation_groups = int(df.loc[station_timestamp_duplicate_mask, duplicate_key_columns].drop_duplicates().shape[0])

print(f"Duplicate station-timestamp rows before aggregation: {duplicate_observation_rows:,}")
print(f"Duplicate station-timestamp groups before aggregation: {duplicate_observation_groups:,}")

if duplicate_observation_rows:
    print("Sample duplicate observations before aggregation:")
    print(df.loc[station_timestamp_duplicate_mask].sort_values(duplicate_key_columns).head(20).to_string())

aggregation_rules = {}
key_set = set(duplicate_key_columns)

for col in df.columns:
    if col in key_set:
        continue
    if col in available_pollutants:
        aggregation_rules[col] = "mean"
    elif col in available_weather_columns:
        aggregation_rules[col] = "mean"
    elif pd.api.types.is_numeric_dtype(df[col]):
        aggregation_rules[col] = "mean"
    else:
        aggregation_rules[col] = "first"

if duplicate_observation_rows:
    rows_before = len(df)
    df = (
        df
        .groupby(duplicate_key_columns, dropna=False, as_index=False)
        .agg(aggregation_rules)
    )
    print(f"Rows removed through duplicate aggregation: {rows_before - len(df):,}")
else:
    print("No station-timestamp aggregation was needed")

remaining_duplicate_observations = int(df.duplicated(subset=duplicate_key_columns).sum())
print(f"Duplicate station-timestamp observations after aggregation: {remaining_duplicate_observations:,}")


Duplicate station-timestamp rows before aggregation: 858
Duplicate station-timestamp groups before aggregation: 429
Sample duplicate observations before aggregation:
                  timestamp       pm2_5        pm10   no         no2   nox        nh3        so2   co     o3  ambient_temperature  relative_humidity  solar_radiation  rainfall    state                  city  latitude  longitude
1085598 2026-03-07 00:00:00   72.250000  150.000000  6.0  194.333333  19.0  15.991283   2.000000  0.0  19.29                27.12             68.155            57.78       0.0  Gujarat  Gidc, Nandesari (, )   22.4069    73.0928
1085599 2026-03-07 00:00:00   72.250000  150.000000  6.0  231.666667  19.0  15.991283   2.000000  0.0  19.29                27.12             68.155            57.78       0.0  Gujarat  Gidc, Nandesari (, )   22.4069    73.0928
1085826 2026-03-14 10:00:00  122.888667  331.355333  6.0   63.544118  19.0  15.991283  30.933333  0.0  19.29                27.12             68.155  

Rows removed through duplicate aggregation: 429
Duplicate station-timestamp observations after aggregation: 0


## 19. Missing Value Analysis


In [19]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": df.isna().mean() * 100,
}).sort_values("missing_pct", ascending=False)

print("Missing values by column:")
print(missing_summary.to_string())

pollutant_missing_summary = pd.DataFrame({
    "missing_count": df[available_pollutants].isna().sum(),
    "missing_pct": df[available_pollutants].isna().mean() * 100,
}).sort_values("missing_pct", ascending=False)

print("\nPollutant missing values after invalid-value cleaning:")
print(pollutant_missing_summary.to_string())

all_pollutants_missing_mask = df[available_pollutants].isna().all(axis=1)
print(f"\nRows where all available pollutants are missing: {all_pollutants_missing_mask.sum():,}")

pollutant_availability_distribution = df[available_pollutants].notna().sum(axis=1).value_counts().sort_index()
print("\nAvailable pollutant count per row distribution:")
print(pollutant_availability_distribution.to_string())


Missing values by column:
                     missing_count  missing_pct
ambient_temperature             10     0.000292
state                            0     0.000000
city                             0     0.000000
solar_radiation                  0     0.000000
relative_humidity                0     0.000000
o3                               0     0.000000
co                               0     0.000000
so2                              0     0.000000
nh3                              0     0.000000
nox                              0     0.000000
no2                              0     0.000000
no                               0     0.000000
pm10                             0     0.000000
pm2_5                            0     0.000000
timestamp                        0     0.000000
longitude                        0     0.000000
latitude                         0     0.000000
rainfall                         0     0.000000

Pollutant missing values after invalid-value cleaning:
      

## 20. Remove Rows Without Any Air-Quality Observation


In [20]:
rows_before = len(df)
df = df.loc[~all_pollutants_missing_mask].copy()
rows_removed = rows_before - len(df)

print(f"Rows removed because all available pollutant values were missing: {rows_removed:,}")
print(f"Rows remaining: {len(df):,}")


Rows removed because all available pollutant values were missing: 0
Rows remaining: 3,429,120


## 21. Missing Pollutant Strategy and Time-Gap Diagnostics

Missing pollutant values are preserved as `NaN` after invalid-value cleaning. This avoids global mean filling, cross-station mixing, and backward filling that would leak future observations into earlier timestamps. Feature engineering and model preparation can choose a train-safe imputation strategy later.


In [21]:
total_missing_pollutant_values = int(df[available_pollutants].isna().sum().sum())
print(f"Total missing pollutant values retained as NaN: {total_missing_pollutant_values:,}")

sort_columns_for_gap_check = STATION_KEYS + ["timestamp"]
df = df.sort_values(sort_columns_for_gap_check).reset_index(drop=True)

if total_missing_pollutant_values:
    station_time_diffs = df.groupby(STATION_KEYS, dropna=False, sort=False)["timestamp"].diff()
    gap_hours = station_time_diffs.dropna().dt.total_seconds() / 3600
    print("Station-wise timestamp gap diagnostics in hours:")
    print(gap_hours.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_string())
    print("\nMost frequent station-wise timestamp gaps in hours:")
    print(gap_hours.round(3).value_counts().head(20).to_string())
else:
    print("No pollutant missing values remain after invalid-value cleaning and removal of pollutant-empty rows.")

print("\nNo pollutant interpolation, forward fill, backward fill, or global mean filling was performed.")


Total missing pollutant values retained as NaN: 0


No pollutant missing values remain after invalid-value cleaning and removal of pollutant-empty rows.

No pollutant interpolation, forward fill, backward fill, or global mean filling was performed.


## 22. Pollutant Distribution Diagnostics


In [22]:
pollutant_statistics = df[available_pollutants].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T
print("Pollutant descriptive statistics after sanity cleaning:")
print(pollutant_statistics.to_string())

print("\nIQR diagnostics only; no IQR-based rows or values are removed.")
iqr_diagnostics = []
for col in available_pollutants:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    upper_fence = q3 + 1.5 * iqr
    iqr_diagnostics.append({
        "pollutant": col,
        "iqr_upper_fence": upper_fence,
        "values_above_iqr_fence": int((df[col] > upper_fence).sum()) if pd.notna(upper_fence) else 0,
    })
print(pd.DataFrame(iqr_diagnostics).to_string(index=False))


Pollutant descriptive statistics after sanity cleaning:
           count        mean         std  min        1%         5%        50%      95%     99%      max
pm2_5  3429120.0   53.906536   60.350993  0.0  2.750000   8.000000  36.165000  156.650  301.00   999.99
pm10   3429120.0  114.714127  106.504706  0.0  9.520000  21.400000  82.750000  318.000  548.50  1000.00
no     3429120.0   13.620381   30.182337  0.0  0.200000   0.932500   6.000000   50.180  142.85   500.00
no2    3429120.0   24.423923   27.292540  0.0  0.375000   2.570000  16.370000   73.940  134.18   499.99
nox    3429120.0   28.203047   36.781768  0.0  0.000000   3.240000  19.000000   82.480  189.40   500.00
nh3    3429120.0   21.168935   22.206830  0.0  0.471111   2.590000  15.991283   56.250  100.70   499.99
co     3429120.0    0.832657    0.840802  0.0  0.000000   0.000000   0.640000    2.220    4.25    41.67
so2    3429120.0   13.056724   15.727464  0.0  0.280000   1.400905   8.820000   37.125   80.40   200.00
o3     3

pollutant  iqr_upper_fence  values_above_iqr_fence
    pm2_5        129.12500                  255238
     pm10        282.14375                  227254
       no         26.64250                  336978
      no2         59.61000                  266189
      nox         66.02500                  245347
      nh3         51.32500                  216470
       co          1.99750                  228640
      so2         31.47500                  249656
       o3         75.39000                  224300


## 23. Final Sort


In [23]:
sort_columns = STATION_KEYS + ["timestamp"]
df = df.sort_values(sort_columns).reset_index(drop=True)

print(f"Final sort columns: {sort_columns}")
print(f"Rows after final sort: {len(df):,}")


Final sort columns: ['state', 'city', 'latitude', 'longitude', 'timestamp']
Rows after final sort: 3,429,120


## 24. Final Validation


In [24]:
assert not df.empty, "Cleaned dataset is empty."
assert pd.api.types.is_datetime64_any_dtype(df["timestamp"]), "timestamp must be datetime dtype."
assert df["timestamp"].notna().all(), "Invalid timestamps remain."

final_pollutant_infinite_count = int(np.isinf(df[available_pollutants]).sum().sum())
assert final_pollutant_infinite_count == 0, "Infinite pollutant values remain."

final_negative_counts = (df[available_pollutants] < 0).sum()
assert int(final_negative_counts.sum()) == 0, "Negative pollutant values remain."

final_above_limit_counts = pd.Series({
    col: int((df[col] > POLLUTANT_MAX_LIMITS[col]).sum())
    for col in available_pollutants
})
assert int(final_above_limit_counts.sum()) == 0, "Pollutant values above configured sanity limits remain."

assert int(df.duplicated().sum()) == 0, "Exact duplicate rows remain."
assert int(df.duplicated(subset=duplicate_key_columns).sum()) == 0, "Duplicate station-timestamp observations remain."

monotonic_by_station = df.groupby(STATION_KEYS, dropna=False, sort=False)["timestamp"].apply(lambda series: series.is_monotonic_increasing)
assert bool(monotonic_by_station.all()), "Station-wise timestamps are not monotonically increasing."

remaining_engineered_columns = [col for col in ENGINEERED_COLUMNS if col in df.columns]
assert not remaining_engineered_columns, f"Engineered columns remain: {remaining_engineered_columns}"

TEMPORARY_HELPER_COLUMNS = ["timestamp_raw", "available_pollutant_count"]
remaining_helper_columns = [col for col in TEMPORARY_HELPER_COLUMNS if col in df.columns]
assert not remaining_helper_columns, f"Temporary helper columns remain: {remaining_helper_columns}"

non_numeric_pollutants = [
    col for col in available_pollutants
    if not pd.api.types.is_numeric_dtype(df[col])
]
assert not non_numeric_pollutants, f"Non-numeric pollutant columns remain: {non_numeric_pollutants}"

for col in ["latitude", "longitude"]:
    if col in df.columns:
        assert pd.api.types.is_numeric_dtype(df[col]), f"{col} must be numeric."

print("Final validation checks passed.")
print(f"Final minimum timestamp: {df['timestamp'].min()}")
print(f"Final maximum timestamp: {df['timestamp'].max()}")
print(f"Available years: {sorted(df['timestamp'].dt.year.dropna().unique().tolist())}")
print(f"Station count: {df[STATION_KEYS].drop_duplicates().shape[0]:,}")

if "state" in df.columns:
    state_names = sorted(df["state"].dropna().unique().tolist())
    print(f"State count: {len(state_names):,}")
    print("State names:")
    print(state_names)

if "city" in df.columns:
    print(f"City count: {df['city'].nunique(dropna=True):,}")

print("\nFinal missing-value summary:")
final_missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": df.isna().mean() * 100,
}).sort_values("missing_pct", ascending=False)
print(final_missing_summary.to_string())

print("\nFinal pollutant descriptive statistics:")
print(df[available_pollutants].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T.to_string())


Final validation checks passed.
Final minimum timestamp: 2017-01-01 00:00:00
Final maximum timestamp: 2026-06-30 23:00:00
Available years: [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
Station count: 66
State count: 19
State names:
['Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chandigarh', 'Chhattisgarh', 'Delhi', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jharkhand', 'Karnataka', 'Kerala', 'Madhya Pradesh', 'Maharashtra', 'Punjab', 'Telangana', 'Uttar Pradesh', 'West Bengal']
City count: 66

Final missing-value summary:
                     missing_count  missing_pct
ambient_temperature             10     0.000292
state                            0     0.000000
city                             0     0.000000
solar_radiation                  0     0.000000
relative_humidity                0     0.000000
o3                               0     0.000000
co                               0     0.000000
so2                              0     0.000000
nh3          

           count        mean         std  min        1%         5%        50%      95%     99%      max
pm2_5  3429120.0   53.906536   60.350993  0.0  2.750000   8.000000  36.165000  156.650  301.00   999.99
pm10   3429120.0  114.714127  106.504706  0.0  9.520000  21.400000  82.750000  318.000  548.50  1000.00
no     3429120.0   13.620381   30.182337  0.0  0.200000   0.932500   6.000000   50.180  142.85   500.00
no2    3429120.0   24.423923   27.292540  0.0  0.375000   2.570000  16.370000   73.940  134.18   499.99
nox    3429120.0   28.203047   36.781768  0.0  0.000000   3.240000  19.000000   82.480  189.40   500.00
nh3    3429120.0   21.168935   22.206830  0.0  0.471111   2.590000  15.991283   56.250  100.70   499.99
co     3429120.0    0.832657    0.840802  0.0  0.000000   0.000000   0.640000    2.220    4.25    41.67
so2    3429120.0   13.056724   15.727464  0.0  0.280000   1.400905   8.820000   37.125   80.40   200.00
o3     3429120.0   27.637983   27.458025  0.0  0.450000   1.9400

## 25. Save Cleaned Parquet Dataset


In [25]:
try:
    import pyarrow as pa
except ModuleNotFoundError:
    print("Parquet write failed before saving because PyArrow is not installed in this environment.")
    print(f"Python version: {sys.version}")
    print(f"pandas version: {pd.__version__}")
    print("PyArrow version: unavailable")
    raise

OUTPUT_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)

try:
    df.to_parquet(
        OUTPUT_FILE,
        index=False,
        engine="pyarrow",
        compression="snappy",
    )
except (pa.ArrowException, TypeError, ValueError) as parquet_error:
    print("Parquet write failed. Environment diagnostics:")
    print(f"Python version: {sys.version}")
    print(f"pandas version: {pd.__version__}")
    print(f"PyArrow version: {pa.__version__}")
    raise parquet_error

parquet_size_bytes = OUTPUT_FILE.stat().st_size
parquet_size_mb = parquet_size_bytes / 1024**2
parquet_size_gb = parquet_size_bytes / 1024**3

print(f"Cleaned Parquet dataset saved to: {OUTPUT_FILE}")
print(f"Final Parquet file size: {parquet_size_mb:,.2f} MB")
print(f"Final Parquet file size: {parquet_size_gb:,.4f} GB")


Cleaned Parquet dataset saved to: ../Data/Interim/cleaned_air_quality.parquet
Final Parquet file size: 94.01 MB
Final Parquet file size: 0.0918 GB


## 26. Final Cleaning Report


In [26]:
station_count = df[STATION_KEYS].drop_duplicates().shape[0]
state_count = df["state"].nunique(dropna=True) if "state" in df.columns else None
city_count = df["city"].nunique(dropna=True) if "city" in df.columns else None
parquet_size_bytes = OUTPUT_FILE.stat().st_size
parquet_size_mb = parquet_size_bytes / 1024**2
parquet_size_gb = parquet_size_bytes / 1024**3

cleaning_operations_performed = [
    "loaded the raw combined AQI_Dataset.csv file",
    "standardized column names to snake_case",
    "standardized descriptive pollutant names to canonical names",
    "removed pre-existing engineered columns",
    "parsed timestamps deterministically from observed raw formats",
    "removed rows with invalid timestamps",
    "converted pollutants, weather columns, and coordinates to numeric values",
    "replaced infinite values with NaN",
    "set negative and above-limit pollutant values to NaN",
    "cleaned state and city text labels and corrected known state spelling issues",
    "validated latitude and longitude against broad India bounds",
    "removed exact duplicate rows",
    "aggregated duplicate station-timestamp observations",
    "removed rows with no available pollutant observation",
    "preserved remaining pollutant NaN values without interpolation or leakage-prone fills",
    "sorted station-wise time-series observations",
    "validated the final cleaned dataset",
    "saved one Snappy-compressed Parquet dataset",
]

print("=" * 80)
print("FINAL DATA CLEANING REPORT")
print("=" * 80)
print(f"Final rows: {len(df):,}")
print(f"Final columns: {df.shape[1]:,}")
print(f"Start timestamp: {df['timestamp'].min()}")
print(f"End timestamp: {df['timestamp'].max()}")
print(f"Number of stations: {station_count:,}")
print(f"Number of states: {state_count:,}" if state_count is not None else "Number of states: state column unavailable")
print(f"Number of cities: {city_count:,}" if city_count is not None else "Number of cities: city column unavailable")
print(f"Available pollutant count: {len(available_pollutants):,}")
print(f"Pollutant columns: {available_pollutants}")
print("Cleaning operations actually performed:")
for operation in cleaning_operations_performed:
    print(f"- {operation}")
print(f"Final Parquet output path: {OUTPUT_FILE}")
print(f"Parquet file size: {parquet_size_mb:,.2f} MB ({parquet_size_gb:,.4f} GB)")
print("=" * 80)


FINAL DATA CLEANING REPORT
Final rows: 3,429,120
Final columns: 18
Start timestamp: 2017-01-01 00:00:00
End timestamp: 2026-06-30 23:00:00
Number of stations: 66
Number of states: 19
Number of cities: 66
Available pollutant count: 9
Pollutant columns: ['pm2_5', 'pm10', 'no', 'no2', 'nox', 'nh3', 'co', 'so2', 'o3']
Cleaning operations actually performed:
- loaded the raw combined AQI_Dataset.csv file
- standardized column names to snake_case
- standardized descriptive pollutant names to canonical names
- removed pre-existing engineered columns
- parsed timestamps deterministically from observed raw formats
- removed rows with invalid timestamps
- converted pollutants, weather columns, and coordinates to numeric values
- replaced infinite values with NaN
- set negative and above-limit pollutant values to NaN
- cleaned state and city text labels and corrected known state spelling issues
- validated latitude and longitude against broad India bounds
- removed exact duplicate rows
- aggregat